# 🏠 House Price Prediction — Ultra Hybrid Training Pipeline

This notebook trains a **5-model stacking ensemble** (XGBoost, LightGBM, GBR, ExtraTrees, Transformer DNN) with an XGBoost meta-learner for Indian house price prediction.

**Pipeline Phases:**
1. Data Loading & Preprocessing
2. Optuna Hyperparameter Tuning
3. Model Initialization
4. Sequential Incremental Training
5. Meta-Learner Stacking
6. Validation & Visualization
7. Save Models & Test Predictions

## 1. Imports & Configuration

In [ ]:
import glob
import logging
import warnings
import os
from pathlib import Path

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

from sklearn.compose import ColumnTransformer
from sklearn.linear_model import Ridge, ElasticNet
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, TargetEncoder
from sklearn.base import BaseEstimator, RegressorMixin, clone
from sklearn.ensemble import GradientBoostingRegressor, ExtraTreesRegressor

import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.layers import (BatchNormalization, Dense, Dropout,
                                     MultiHeadAttention, LayerNormalization,
                                     GlobalAveragePooling1D, Reshape)
from tensorflow.keras.models import Sequential

from lightgbm import LGBMRegressor
from xgboost import XGBRegressor

warnings.filterwarnings("ignore")

DATA_DIR = Path("Training_dataset")
MODEL_OUTPUT_PATH = Path("custom_hybrid_model.pkl")
TEST_SIZE = 0.2
RANDOM_STATE = 42

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-7s | %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)

## 2. Utility Functions

In [ ]:
def detect_gpu() -> bool:
    import subprocess
    try:
        result = subprocess.run(["nvidia-smi"], capture_output=True, text=True, timeout=5)
        if result.returncode == 0:
            log.info("NVIDIA GPU detected - algorithms will use CUDA where possible.")
            return True
    except Exception:
        pass
    log.info("No NVIDIA GPU detected - running on CPU.")
    return False


def get_train_files() -> list:
    train_files = sorted(glob.glob(str(DATA_DIR / "train_part*.csv")))
    if not train_files:
        raise FileNotFoundError(f"No train_part*.csv files found in {DATA_DIR}")
    return train_files

## 3. Preprocessor

In [ ]:
def initialize_preprocessor(X: pd.DataFrame) -> ColumnTransformer:
    cols_to_exclude = ['ListingID', 'RERAID']
    features = [c for c in X.columns if c not in cols_to_exclude]

    numeric_features = X[features].select_dtypes(include=['int64', 'float64']).columns.tolist()
    categorical_features = X[features].select_dtypes(include=['object', 'category']).columns.tolist()

    log.info("Numeric Features (%d)", len(numeric_features))
    log.info("Categorical Features (%d)", len(categorical_features))

    numeric_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])

    categorical_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('target_encoder', TargetEncoder(target_type='continuous', random_state=RANDOM_STATE)),
        ('scaler', StandardScaler())
    ])

    return ColumnTransformer(
        transformers=[
            ('num', numeric_transformer, numeric_features),
            ('cat', categorical_transformer, categorical_features)
        ],
        remainder='drop'
    )

## 4. Transformer + ResNet DNN Architecture

In [ ]:
def build_transformer_dnn(input_dim: int, units1: int = 512, units2: int = 256,
                           units3: int = 128, dropout_rate: float = 0.3,
                           lr: float = 0.001) -> tf.keras.Model:
    inputs = tf.keras.Input(shape=(input_dim,))

    x = Dense(units1, activation='swish')(inputs)
    x = BatchNormalization()(x)
    x = Dropout(dropout_rate)(x)

    res = Dense(units2, activation='swish')(x)
    res = BatchNormalization()(res)
    res = Dropout(dropout_rate * 0.75)(res)
    res = Dense(units2, activation='swish')(res)
    res = BatchNormalization()(res)

    x_proj = Dense(units2)(x)
    x = tf.keras.layers.Add()([x_proj, res])
    x = tf.keras.layers.Activation('swish')(x)

    seq = Reshape((1, input_dim))(inputs)
    d_model = 64
    seq = Dense(d_model)(seq)
    attn_out = MultiHeadAttention(num_heads=4, key_dim=16)(seq, seq)
    attn_out = LayerNormalization()(attn_out + seq)
    attn_flat = GlobalAveragePooling1D()(attn_out)

    merged = tf.keras.layers.Concatenate()([x, attn_flat])
    merged = Dense(units3, activation='swish')(merged)
    merged = BatchNormalization()(merged)
    merged = Dropout(dropout_rate * 0.5)(merged)
    merged = Dense(64, activation='swish')(merged)

    output = Dense(1, activation='linear')(merged)

    model = tf.keras.Model(inputs=inputs, outputs=output)
    optimizer = tf.keras.optimizers.AdamW(learning_rate=lr, weight_decay=1e-4)
    model.compile(optimizer=optimizer, loss='huber', metrics=['mae'])
    return model

## 5. Optuna Hyperparameter Tuning

In [ ]:
def tune_xgboost(X_tr, y_tr, use_gpu: bool, n_trials: int = 30) -> dict:
    log.info("Running Optuna search for XGBoost (%d trials)...", n_trials)
    def objective(trial):
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 1000, 3000),
            'max_depth': trial.suggest_int('max_depth', 10, 15),
            'learning_rate': trial.suggest_float('learning_rate', 0.003, 0.05, log=True),
            'subsample': trial.suggest_float('subsample', 0.6, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
            'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
            'gamma': trial.suggest_float('gamma', 0.0, 1.0),
            'reg_alpha': trial.suggest_float('reg_alpha', 1e-5, 1.0, log=True),
            'reg_lambda': trial.suggest_float('reg_lambda', 1e-5, 1.0, log=True),
            'verbosity': 0,
            'random_state': RANDOM_STATE,
        }
        if use_gpu:
            params['device'] = 'cuda'
            params['tree_method'] = 'hist'
        X_np = np.array(X_tr) if not isinstance(X_tr, np.ndarray) else X_tr
        y_np = np.array(y_tr) if not isinstance(y_tr, np.ndarray) else y_tr
        kf = KFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
        cv_scores = []
        for tr_idx, val_idx in kf.split(X_np):
            model = XGBRegressor(**params)
            model.fit(X_np[tr_idx], y_np[tr_idx])
            preds = model.predict(X_np[val_idx])
            r2 = r2_score(y_np[val_idx], np.expm1(preds))
            cv_scores.append(r2)
        return np.mean(cv_scores)
    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)
    log.info("Best XGBoost R2 (CV): %.4f | Params: %s", study.best_value, study.best_params)
    return study.best_params


def tune_lgbm(X_tr, y_tr, use_gpu: bool, n_trials: int = 30) -> dict:
    log.info("Running Optuna search for LightGBM (%d trials)...", n_trials)
    def objective(trial):
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 1000, 3000),
            'max_depth': -1,
            'num_leaves': trial.suggest_int('num_leaves', 127, 1023),
            'learning_rate': trial.suggest_float('learning_rate', 0.003, 0.05, log=True),
            'subsample': trial.suggest_float('subsample', 0.6, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
            'min_child_samples': trial.suggest_int('min_child_samples', 5, 50),
            'reg_alpha': trial.suggest_float('reg_alpha', 1e-5, 1.0, log=True),
            'reg_lambda': trial.suggest_float('reg_lambda', 1e-5, 1.0, log=True),
            'random_state': RANDOM_STATE,
            'device_type': 'gpu' if use_gpu else 'cpu',
            'verbose': -1,
        }
        X_np = np.array(X_tr) if not isinstance(X_tr, np.ndarray) else X_tr
        y_np = np.array(y_tr) if not isinstance(y_tr, np.ndarray) else y_tr
        kf = KFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
        cv_scores = []
        for tr_idx, val_idx in kf.split(X_np):
            model = LGBMRegressor(**params)
            model.fit(X_np[tr_idx], y_np[tr_idx])
            preds = model.predict(X_np[val_idx])
            r2 = r2_score(y_np[val_idx], np.expm1(preds))
            cv_scores.append(r2)
        return np.mean(cv_scores)
    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)
    log.info("Best LightGBM R2 (CV): %.4f | Params: %s", study.best_value, study.best_params)
    return study.best_params

## 6. Load Data & Initialize

In [ ]:
target_col = "Price_INR"
train_files = get_train_files()
print(f"Files to train on: {[Path(f).name for f in train_files]}")

first_df = pd.read_csv(train_files[0], on_bad_lines='skip').dropna(axis=1, how='all')
first_df = first_df.dropna(subset=[target_col])
X_init = first_df.drop(columns=[target_col])

print(f"Dataset shape: {first_df.shape}")
first_df.head()

## 7. Fit Preprocessor & Split Data

In [ ]:
preprocessor = initialize_preprocessor(X_init)

X_train_chunk, X_val, y_train_chunk, y_val = train_test_split(
    X_init, first_df[target_col], test_size=TEST_SIZE, random_state=RANDOM_STATE
)

preprocessor.fit(X_train_chunk, y_train_chunk)
X_val_processed = preprocessor.transform(X_val)
input_dim = X_val_processed.shape[1]
print(f"Final feature dimension: {input_dim}")

use_gpu = detect_gpu()

## 8. PHASE 1 — Optuna Hyperparameter Search

In [ ]:
X_tr_log = preprocessor.transform(X_train_chunk)
y_tr_log = np.log1p(y_train_chunk)

best_xgb_params = tune_xgboost(X_tr_log, y_tr_log, use_gpu, n_trials=30)
best_lgb_params = tune_lgbm(X_tr_log, y_tr_log, use_gpu, n_trials=30)

## 9. PHASE 2 — Initialize Base Learners

In [ ]:
best_xgb_params['random_state'] = RANDOM_STATE
best_xgb_params['verbosity'] = 0
if use_gpu:
    best_xgb_params['device'] = 'cuda'
    best_xgb_params['tree_method'] = 'hist'

best_lgb_params['random_state'] = RANDOM_STATE
best_lgb_params['verbose'] = -1
best_lgb_params['device_type'] = 'gpu' if use_gpu else 'cpu'

model_xgb = XGBRegressor(**best_xgb_params)
model_lgb = LGBMRegressor(**best_lgb_params)

model_gbr = GradientBoostingRegressor(
    n_estimators=2000, learning_rate=0.005, max_depth=10,
    subsample=0.8, random_state=RANDOM_STATE, verbose=0
)

model_et = ExtraTreesRegressor(
    n_estimators=1000, max_depth=None,
    random_state=RANDOM_STATE, n_jobs=-1
)

model_dnn = build_transformer_dnn(input_dim)
model_dnn.summary()

## 10. PHASE 3 — Sequential Incremental Training

In [ ]:
meta_xgb_kwargs = {
    'n_estimators': 500, 'max_depth': 5, 'learning_rate': 0.05,
    'random_state': RANDOM_STATE, 'verbosity': 0
}
if use_gpu:
    meta_xgb_kwargs['device'] = 'cuda'
    meta_xgb_kwargs['tree_method'] = 'hist'
meta_learner = XGBRegressor(**meta_xgb_kwargs)
oof_meta_features = []
global_y_train_log = []
xgb_trained = None
lgb_trained = None

for idx, f in enumerate(train_files):
    print(f"\n--- Processing Dataset {idx+1}/{len(train_files)}: {Path(f).name} ---")
    if idx == 0:
        X_tr, y_tr = X_train_chunk, y_train_chunk
    else:
        df = pd.read_csv(f, on_bad_lines='skip').dropna(axis=1, how='all')
        df = df.dropna(subset=[target_col])
        y_tr = df[target_col]
        X_tr = df.drop(columns=[target_col])

    X_tr_p = preprocessor.transform(X_tr)
    y_tr_log = np.log1p(y_tr).values if hasattr(y_tr, 'values') else np.log1p(y_tr)

    if xgb_trained is None:
        model_xgb.fit(X_tr_p, y_tr_log)
    else:
        model_xgb.fit(X_tr_p, y_tr_log, xgb_model=model_xgb.get_booster())
    xgb_trained = True

    if lgb_trained is None:
        model_lgb.fit(X_tr_p, y_tr_log)
    else:
        model_lgb.fit(X_tr_p, y_tr_log, init_model=model_lgb.booster_)
    lgb_trained = True

    model_gbr.fit(X_tr_p, y_tr_log)
    model_et.fit(X_tr_p, y_tr_log)

    callbacks = [
        EarlyStopping(monitor='val_loss', patience=150, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.4, patience=30, verbose=1, min_lr=1e-7)
    ]
    model_dnn.fit(X_tr_p, y_tr_log, epochs=2000, batch_size=16,
                  validation_split=0.15, callbacks=callbacks, verbose=1)

    xgb_p = model_xgb.predict(X_tr_p)
    lgb_p = model_lgb.predict(X_tr_p)
    gbr_p = model_gbr.predict(X_tr_p)
    et_p = model_et.predict(X_tr_p)
    dnn_p = model_dnn.predict(X_tr_p, verbose=0).flatten()
    chunk_meta = np.column_stack((xgb_p, lgb_p, gbr_p, et_p, dnn_p))
    oof_meta_features.append(chunk_meta)
    global_y_train_log.extend(y_tr_log)

print("\nAll datasets processed!")

## 11. PHASE 4 — Train Meta-Learner

In [ ]:
X_meta = np.vstack(oof_meta_features)
y_meta_log = np.array(global_y_train_log)
meta_learner.fit(X_meta, y_meta_log)
print("Meta-learner trained on stacked predictions.")

## 12. PHASE 5 — Validation Metrics

In [ ]:
val_xgb = model_xgb.predict(X_val_processed)
val_lgb = model_lgb.predict(X_val_processed)
val_gbr = model_gbr.predict(X_val_processed)
val_et  = model_et.predict(X_val_processed)
val_dnn = model_dnn.predict(X_val_processed, verbose=0).flatten()

val_meta = np.column_stack((val_xgb, val_lgb, val_gbr, val_et, val_dnn))
preds_log = meta_learner.predict(val_meta)
preds = np.expm1(preds_log)

r2   = r2_score(y_val, preds)
mae  = mean_absolute_error(y_val, preds)
rmse = np.sqrt(mean_squared_error(y_val, preds))

print(f"R2 Score : {r2:.4f}")
print(f"MAE      : {mae:.2f}")
print(f"RMSE     : {rmse:.2f}")

## 13. Visualization — Actual vs Predicted

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(x=y_val, y=preds, alpha=0.5)
max_v = max(y_val.max(), preds.max())
min_v = min(y_val.min(), preds.min())
plt.plot([min_v, max_v], [min_v, max_v], color='red', linestyle='--')
plt.title('Ultra Hybrid Model: Actual vs Predicted Prices')
plt.xlabel('Actual Price (INR)')
plt.ylabel('Predicted Price (INR)')
plt.tight_layout()
plt.savefig("hybrid_actual_vs_predicted.png")
plt.show()

## 14. Visualization — 3D Residual Plot

In [ ]:
fig = plt.figure(figsize=(12, 8))
ax = fig.add_subplot(111, projection='3d')
residuals = np.abs(np.array(y_val) - preds)
ax.scatter(y_val, preds, residuals, c=residuals, cmap='plasma', alpha=0.6)
ax.set_xlabel('Actual Price (INR)')
ax.set_ylabel('Predicted Price (INR)')
ax.set_zlabel('Absolute Error')
ax.set_title('3D Plot: Predicted vs Actual vs Residual Error')
plt.savefig("hybrid_3d_residuals.png")
plt.show()

## 15. Save Models

In [ ]:
final_artifact = {
    'preprocessor': preprocessor,
    'model_xgb': model_xgb,
    'model_lgb': model_lgb,
    'model_gbr': model_gbr,
    'model_et': model_et,
    'meta_learner': meta_learner,
    'xgb_params': best_xgb_params,
    'lgb_params': best_lgb_params,
}
joblib.dump(final_artifact, MODEL_OUTPUT_PATH)
model_dnn.save("custom_hybrid_keras.keras")
print(f"Saved models to {MODEL_OUTPUT_PATH}")

## 16. Generate Test Predictions

In [ ]:
test_files = glob.glob(str(DATA_DIR / "test_part*.csv"))
if test_files:
    test_dfs = [pd.read_csv(f, on_bad_lines='skip') for f in test_files]
    test_df = pd.concat(test_dfs, ignore_index=True).dropna(axis=1, how='all')
    print(f"Test dataset shape: {test_df.shape}")
    X_test_p = preprocessor.transform(test_df)

    tst_xgb = model_xgb.predict(X_test_p)
    tst_lgb = model_lgb.predict(X_test_p)
    tst_gbr = model_gbr.predict(X_test_p)
    tst_et  = model_et.predict(X_test_p)
    tst_dnn = model_dnn.predict(X_test_p, verbose=0).flatten()

    tst_meta = np.column_stack((tst_xgb, tst_lgb, tst_gbr, tst_et, tst_dnn))
    test_preds = np.expm1(meta_learner.predict(tst_meta))

    submission_df = pd.DataFrame()
    if 'ListingID' in test_df.columns:
        submission_df['ListingID'] = test_df['ListingID']
    submission_df['Predicted_Price_INR'] = test_preds
    submission_df.to_csv("test_predictions.csv", index=False)
    print("Saved test predictions to test_predictions.csv")
    submission_df.head()
else:
    print("No test datasets found.")